# Wan2.1 (1.3B): real computer monitor in a dark server room (green screen)

In [ ]:
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
import torch; print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
import torch, time, os
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
MODEL = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()   # T4'ün 16 GB belleğine sığdırır
print('model hazır')

In [ ]:
import os, time, torch
assert 'pipe' in globals(), 'Önce üstteki model hücresini çalıştır (en kolayı: Çalışma zamanı → Tümünü çalıştır)'
SCENES = {
    "monitor": (
        "Realistic cinematic shot, slow dolly-out from a single modern computer monitor standing on a desk in a dark server room at night. "
        "The monitor screen is a perfectly flat, uniform, bright chroma-key green, filling the whole screen, no image on it. "
        "Behind it, rows of tall black server racks with many tiny blinking blue and green LED lights, faint haze, cold blue ambient light, "
        "the green glow of the screen reflecting on the desk. Real footage, photorealistic, sharp focus, shallow depth of field, 35mm film"
    ),
}
NEG = ("cartoon, anime, 3d render, illustration, text, watermark, logo, subtitles, blurry, low quality, jpeg artifacts, "
       "deformed, extra limbs, extra fingers, static frame, worst quality, human clothing")
NEG_SCENE = {"monitor": ", cartoon, 3d render, CGI, video game, text on screen, logo, people, hands, robot, multiple monitors, curved screen, pattern on screen", "darkspark": ", campfire, flames, burning fire, torch, fireball, fireworks, magic sparkles, glitter, cartoon sparks, lightning, explosion, CGI particles, daylight, blue tint, glowing eyes", "snowspark": ", fire, campfire, flames, torch, fireworks, magic sparkles, glitter, cartoon sparks, lightning, explosion, CGI particles, daylight, green grass", "robot": ", cartoon, toy, human hand, skin, famous robot, logo", "gunner": ", cartoon smoke, extra fingers, blurry face", "salvo": ", cartoon, fireworks, single rocket", "approach": "", "strike": ", dark, underexposed, murky, gloomy, fireworks, magic sparkles, glitter, cartoon sparks, lightning, explosion, CGI particles"}

W, H, FRAMES = 480, 832, 81   # dikey, 81 kare yaklaşık 5 sn (16 fps)
SEEDS = [42]               # her sahneden 2 farklı deneme, en iyisini seçeriz
os.makedirs('clips', exist_ok=True)
TOTAL = len(SCENES) * len(SEEDS); done = 0
for name, prompt in SCENES.items():
    for seed in SEEDS:
        print(f'▶ {name} (seed {seed}) üretiliyor…  Genel ilerleme: {done}/{TOTAL} klip – %{round(100*done/TOTAL)}')
        t0 = time.time()
        out = pipe(prompt=prompt, negative_prompt=NEG + NEG_SCENE[name], width=W, height=H, num_frames=FRAMES,
                   guidance_scale=5.0, num_inference_steps=30,
                   generator=torch.Generator('cpu').manual_seed(seed)).frames[0]
        export_to_video(out, f'clips/{name}_seed{seed}.mp4', fps=16)
        done += 1
        print(f'✅ {name} (seed {seed}) bitti: {round((time.time()-t0)/60, 1)} dk  |  Genel ilerleme: {done}/{TOTAL} klip – %{round(100*done/TOTAL)}')

Daha fazla deneme istersen `SEEDS` listesine başka sayılar ekle (örn. `[42, 7, 123]`).

In [ ]:
import shutil; shutil.make_archive('monitor_clips', 'zip', 'clips')
try:
    from google.colab import files; files.download('monitor_clips.zip')
except ImportError:
    print('Kaggle: sağdaki Output bölümünden monitor_clips.zip dosyasını indir')